# 🔄 Interview Questions: ETL & Data Pipeline Design
## From Data Extraction to Production-Ready Pipelines

### 🎯 Why ETL Questions Define Data Engineering Roles

**ETL isn't just about moving data—it's about building reliable, scalable systems.** Here's why ETL mastery matters:

1. **Core Responsibility** - 70% of data engineering work is pipeline design and maintenance
2. **System Design Signal** - Shows you can architect end-to-end data flows
3. **Production Readiness** - Separates those who've built pipelines from those who've read about them
4. **Business Impact** - Bad pipelines = bad data = bad decisions = lost revenue
5. **Interview Depth** - ETL questions probe design thinking, not just SQL syntax

### 💡 What Separates Junior from Senior Data Engineers

| Junior Data Engineer | Senior Data Engineer |
|---------------------|----------------------|
| Writes one-off scripts | Designs reusable, scheduled pipelines |
| "My query works" | "My pipeline handles failures gracefully" |
| Full loads every time | Incremental loads with watermarks |
| No data validation | Schema validation, quality checks, alerts |
| Hardcoded connections | Config-driven, environment-aware |
| Manual reruns | Idempotent, self-healing pipelines |
| "It failed, I'll check tomorrow" | Monitoring, alerting, SLAs |

---

### 📊 Interview Question Coverage (20 Questions)

This module covers **7 critical ETL domains**:

| Topic | Questions | Why It Matters |
|-------|-----------|----------------|
| **ETL Design Patterns** | 3 | Full load, incremental, CDC, microbatch |
| **Data Quality & Validation** | 3 | Schema checks, data profiling, anomaly detection |
| **Error Handling & Recovery** | 3 | Idempotency, checkpoints, dead letter queues |
| **Scheduling & Orchestration** | 2 | DAGs, dependencies, backfills |
| **Performance & Optimization** | 3 | Parallelization, partitioning, bottleneck analysis |
| **Data Lineage & Metadata** | 2 | Tracking data sources, transformations |
| **Testing & Monitoring** | 4 | Unit tests, integration tests, SLA monitoring |

---

### 🎓 How to Master This Module

1. **Think end-to-end** - Source → Extract → Transform → Load → Validate → Monitor
2. **Design for failure** - Networks fail, APIs timeout, data arrives late
3. **Make it idempotent** - Running twice = same result as running once
4. **Track everything** - Watermarks, checksums, row counts, timestamps
5. **Automate validation** - Don't trust source data, verify everything

### 🏆 Interview Success Tips

✅ **Ask clarifying questions** - "What's the data volume? SLA? Failure scenarios?"
✅ **Design for scale** - "This works for 1M rows, but for 1B rows I'd..."
✅ **Mention monitoring** - "I'd add alerts for..."
✅ **Discuss trade-offs** - "Real-time vs batch, complexity vs speed"
✅ **Show production awareness** - "Late data, schema changes, backfills"

⚠️ **Red flags that fail interviews:**
- Doesn't consider failure scenarios
- No mention of data validation
- Can't explain idempotency
- Never heard of watermarks or checkpoints
- No monitoring or alerting strategy
- Designs pipelines that aren't rerunnable
- Ignores data quality issues

---

**Ready to master ETL and data pipeline design? Let's dive in!** 🚀

## 🏗️ Section 1: ETL Design Patterns (3 Questions)

Choosing the right ETL pattern determines pipeline efficiency, reliability, and maintainability.

### ❓ Question 1: Full Load vs Incremental Load Design

**Fundamental ETL Question:**
> "You need to sync a 10 billion row customer orders table from MySQL to a data warehouse. The table has an updated_at timestamp. The initial load must happen once, then daily updates. Design both the initial full load and the incremental pipeline. What's your strategy for handling late-arriving updates?"

### ✅ Answer 1: Full Load vs Incremental Load Strategies

#### **Pattern Comparison:**

| Aspect | Full Load | Incremental Load |
|--------|-----------|------------------|
| **Frequency** | Once, or infrequent | Daily/hourly/real-time |
| **Data Scanned** | Entire table | Only new/changed rows |
| **Complexity** | Simple | Complex (watermarks, deduplication) |
| **Performance** | Slow (reads everything) | Fast (reads subset) |
| **Consistency** | Guaranteed | Requires careful design |
| **Use Case** | Dimensions, small tables | Facts, large tables |

---

#### **Full Load Implementation:**

**When to Use:**
- Small tables (< 10M rows)
- Dimension tables (slow-changing)
- Initial one-time loads
- Data freshness not critical (weekly/monthly)

**Implementation:**

```sql
-- Step 1: Extract from source (MySQL)
SELECT * FROM source_db.orders;

-- Step 2: Load strategy
-- Option A: Truncate and reload
TRUNCATE TABLE target.orders;
INSERT INTO target.orders SELECT * FROM staging.orders;

-- Option B: Replace entire table (safer, atomic)
CREATE TABLE target.orders_new AS SELECT * FROM staging.orders;
DROP TABLE target.orders;
ALTER TABLE target.orders_new RENAME TO orders;

-- Option C: MERGE (update existing, insert new)
MERGE INTO target.orders AS t
USING staging.orders AS s
ON t.order_id = s.order_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;
```

**Pros:**
✅ Simple logic (no watermarks)
✅ Always consistent (no missed updates)
✅ No duplicate handling needed

**Cons:**
❌ Slow (reads entire source table)
❌ High resource usage (network, storage, compute)
❌ Locks source table (if using snapshot)

---

#### **Incremental Load Implementation:**

**When to Use:**
- Large tables (> 100M rows)
- Frequent updates (hourly/daily)
- Near real-time requirements
- Cost optimization (process less data)

**Key Components:**

**1. Watermark Table (Track Progress):**
```sql
CREATE TABLE etl_watermarks (
  source_table STRING,
  last_processed_timestamp TIMESTAMP,
  last_processed_id BIGINT,
  updated_at TIMESTAMP,
  rows_processed BIGINT
);

-- Initialize watermark
INSERT INTO etl_watermarks VALUES (
  'orders',
  '1970-01-01 00:00:00',  -- Start from beginning
  0,
  CURRENT_TIMESTAMP,
  0
);
```

**2. Incremental Extract:**
```sql
-- Get watermark
SET watermark = (
  SELECT last_processed_timestamp 
  FROM etl_watermarks 
  WHERE source_table = 'orders'
);

-- Extract only new/changed data
SELECT *
FROM source_db.orders
WHERE updated_at > watermark
  AND updated_at <= CURRENT_TIMESTAMP;  -- Upper bound for consistency
```

**3. Load to Target:**
```sql
-- Use MERGE for idempotency (handles duplicates)
MERGE INTO target.orders AS t
USING staging.incremental_orders AS s
ON t.order_id = s.order_id
WHEN MATCHED AND s.updated_at > t.updated_at THEN
  UPDATE SET 
    t.customer_id = s.customer_id,
    t.order_date = s.order_date,
    t.amount = s.amount,
    t.status = s.status,
    t.updated_at = s.updated_at
WHEN NOT MATCHED THEN
  INSERT (order_id, customer_id, order_date, amount, status, updated_at)
  VALUES (s.order_id, s.customer_id, s.order_date, s.amount, s.status, s.updated_at);
```

**4. Update Watermark:**
```sql
UPDATE etl_watermarks
SET 
  last_processed_timestamp = (
    SELECT MAX(updated_at) FROM staging.incremental_orders
  ),
  updated_at = CURRENT_TIMESTAMP,
  rows_processed = (
    SELECT COUNT(*) FROM staging.incremental_orders
  )
WHERE source_table = 'orders';
```

---

#### **Handling Late-Arriving Data:**

**Problem:** Data for yesterday arrives today due to source system delays.

**Solution 1: Lookback Window**
```sql
-- Process last 3 days (1-day lookback buffer)
SELECT *
FROM source_db.orders
WHERE updated_at >= watermark - INTERVAL 1 DAY
  AND updated_at <= CURRENT_TIMESTAMP;

-- MERGE handles duplicates automatically
```

**Solution 2: Separate Late Data Pipeline**
```sql
-- Main pipeline: Process recent data
WHERE updated_at >= watermark
  AND updated_at <= CURRENT_TIMESTAMP;

-- Late data pipeline (runs hourly): Check last 7 days
WHERE updated_at >= CURRENT_TIMESTAMP - INTERVAL 7 DAY
  AND order_id NOT IN (
    SELECT order_id FROM target.orders WHERE updated_at >= CURRENT_TIMESTAMP - INTERVAL 7 DAY
  );
```

**Solution 3: Event Time vs Processing Time**
```sql
CREATE TABLE orders (
  order_id BIGINT,
  event_time TIMESTAMP,      -- When order was created
  processing_time TIMESTAMP,  -- When we received it
  ingestion_time TIMESTAMP    -- When we loaded it to warehouse
);

-- Track all three timestamps for late data analysis
```

---

#### **Hybrid Pattern: Initial Full + Incremental:**

**Day 1: Full Load**
```python
# Extract all historical data
full_load_query = "SELECT * FROM orders WHERE created_at < '2024-01-01'"

# Load to target
load_to_warehouse(full_load_query, mode='overwrite')

# Set watermark
set_watermark('orders', '2024-01-01 00:00:00')
```

**Day 2+: Incremental**
```python
# Get watermark
watermark = get_watermark('orders')

# Extract incremental
incremental_query = f"""
  SELECT * FROM orders 
  WHERE updated_at > '{watermark}'
  AND updated_at <= CURRENT_TIMESTAMP
"""

# Merge to target
merge_to_warehouse(incremental_query)

# Update watermark
update_watermark('orders', max_updated_at)
```

---

#### **Partitioned Incremental Load:**

```sql
-- For very large tables, process in date partitions
WITH unprocessed_dates AS (
  SELECT DISTINCT DATE(updated_at) AS partition_date
  FROM source_db.orders
  WHERE updated_at > watermark
)
SELECT 
  o.*,
  d.partition_date
FROM source_db.orders o
INNER JOIN unprocessed_dates d 
  ON DATE(o.updated_at) = d.partition_date
ORDER BY partition_date;  -- Process oldest first
```

**Process each partition separately:**
```python
for partition_date in unprocessed_dates:
    # Extract partition
    data = extract_partition(partition_date)
    
    # Load partition
    load_partition(data, partition_date)
    
    # Checkpoint
    mark_partition_complete(partition_date)
```

---

#### **Change Data Capture (CDC) Pattern:**

**Best for:** Real-time or near-real-time pipelines

```sql
-- Source system provides CDC feed
CREATE TABLE orders_cdc (
  order_id BIGINT,
  operation STRING,  -- 'I', 'U', 'D' (Insert/Update/Delete)
  sequence_num BIGINT,
  commit_timestamp TIMESTAMP,
  -- ... order columns
);

-- Apply CDC changes
MERGE INTO target.orders AS t
USING (
  SELECT *
  FROM orders_cdc
  WHERE sequence_num > last_processed_sequence
  ORDER BY sequence_num
) AS s
ON t.order_id = s.order_id
WHEN MATCHED AND s.operation = 'U' THEN
  UPDATE SET *
WHEN MATCHED AND s.operation = 'D' THEN
  DELETE
WHEN NOT MATCHED AND s.operation = 'I' THEN
  INSERT *;
```

---

#### **Idempotency Checklist:**

✅ Use MERGE instead of INSERT
✅ Use unique keys for deduplication
✅ Track watermarks in transactions
✅ Handle duplicates in source data
✅ Test: Run pipeline twice, verify same result

---

#### **Interview Follow-Up:**

**Q: "What if the source table doesn't have updated_at?"**

**A:** Options:
1. Request source system add audit column
2. Use hash/checksum to detect changes (expensive)
3. Compare entire table (very expensive)
4. Use CDC if available
5. Fall back to full load

**Q: "How do you handle deletes in incremental load?"**

**A:**
1. **Soft delete:** Source marks is_deleted=TRUE, we capture in incremental
2. **CDC:** Capture DELETE operations explicitly
3. **Full reconciliation:** Periodic comparison to find missing records
4. **Tombstone records:** Source inserts delete record with NULL values

**Q: "What if incremental load fails halfway?"**

**A:**
- Use transactions (atomic watermark update)
- Checkpoint progress (batch-level watermarks)
- Make pipeline idempotent (rerun = same result)
- Don't update watermark until ALL data loaded successfully